In [28]:
from docx import Document
import glob

from IPython.display import Markdown, display
import ollama
import time
import os
import re

### Now we write a function that gets the text from docx files and returns it as a string.

In [29]:
def extract_text_from_docx(file_path):
    """Extract text from a .docx file, including paragraphs and tables."""
    doc = Document(file_path)
    text = []

    # Extract paragraphs
    for paragraph in doc.paragraphs:
        text.append(paragraph.text)

    # Extract text from tables
    for table in doc.tables:
        for row in table.rows:
            for cell in row.cells:
                for paragraph in cell.paragraphs:
                    text.append(paragraph.text)

    return '\n'.join(text)

def read_folder(folder_path):
    folder_path = f'{folder_path}/*.docx'

    object = {}
    for file_path in glob.glob(folder_path):
        text = extract_text_from_docx(file_path)
        file_path = file_path.replace(f"DataSet/{os.path.basename(os.path.dirname(file_path))}\\", ' ')
        object[file_path] = text

    return object

In [30]:
cvs = read_folder('DataSet/cv')
jobs = read_folder('DataSet/job_descriptions')

In [31]:
display(Markdown(f"**Number of CVs:** {len(cvs)}"))
for key, value in cvs.items():
    display(Markdown(f"**{key}**"))
    display(Markdown(f"{value}"))
    break

**Number of CVs:** 500

** cv_100_Elena_Andreea_Vâlceanu.docx**

Elena Andreea Vâlceanu
Technical Skills
Python, TensorFlow
JavaScript, ReactJS
AWS SageMaker, Docker
SQL, PostgreSQL
Figma, Adobe XD
Foreign Languages
- English: C1
- Spanish: B1
Education
- University Name: University Politehnica of Bucharest
- Program Duration: 4 years
- Master Degree Name: University Politehnica of Bucharest
- Program Duration: 2 years
Certifications
- AWS Certified Machine Learning – Specialty
- Docker Certified Associate
- TensorFlow Developer Certificate
Project Experience
1. Predictive Analytics Platform
   Developed a predictive analytics platform using Python and TensorFlow, aimed at providing real-time insights for retail businesses. Leveraged AWS SageMaker for model training and deployment, ensuring scalable and efficient processing of large datasets. Implemented Docker containers to streamline the development and deployment process, enhancing collaboration across teams. Technologies and tools used: Python, TensorFlow, AWS SageMaker, Docker.

2. Interactive Web Application for Data Visualization
   Created an interactive web application for visualizing complex datasets using JavaScript and ReactJS. Designed intuitive user interfaces with Figma and Adobe XD, focusing on enhancing user engagement and accessibility. Integrated PostgreSQL for robust data management and implemented SQL queries to optimize data retrieval processes. Technologies and tools used: JavaScript, ReactJS, Figma, Adobe XD, SQL, PostgreSQL.

In [32]:
display(Markdown(f"**Number of jobs:** {len(jobs)}"))
for key, value in jobs.items():
    display(Markdown(f"**{key}**"))
    display(Markdown(f"{value}"))
    break

**Number of jobs:** 100

** job_description_100_UIUX Designer.docx**

Job Title:
Senior UI/UX Designer
Company Overview:
InnovateTech Solutions is a leading technology company dedicated to creating cutting-edge digital products that enhance user experiences across various platforms. Our team is passionate about innovation, creativity, and delivering exceptional solutions that meet the evolving needs of our clients. We pride ourselves on fostering a collaborative and inclusive work environment where every team member's ideas are valued and contribute to our success.
Key Responsibilities:
- Lead the design and development of user interfaces and experiences for web and mobile applications, ensuring a seamless and intuitive user journey.
- Collaborate with cross-functional teams, including product managers, developers, and other designers, to translate business requirements into innovative design solutions.
- Conduct user research and usability testing to gather insights and validate design concepts, iterating based on feedback to enhance user satisfaction.
- Create wireframes, prototypes, and high-fidelity designs using industry-standard design tools, ensuring consistency with brand guidelines and design systems.
- Mentor and provide guidance to junior designers, fostering a culture of continuous learning and improvement within the design team.
- Stay updated with the latest UI/UX trends, techniques, and technologies, and apply them to improve design processes and deliverables.
- Present design concepts and solutions to stakeholders, articulating design rationale and incorporating feedback to refine designs.
Required Qualifications:
- Bachelor’s degree in Design, Human-Computer Interaction, or a related field.
- Minimum of 5 years of experience in UI/UX design, with a strong portfolio showcasing diverse design projects.
- Proficiency in design software such as Adobe Creative Suite, Sketch, Figma, or similar tools.
- Strong understanding of user-centered design principles and best practices.
- Excellent communication and presentation skills, with the ability to articulate design decisions effectively.
Preferred Skills:
- Experience with front-end development technologies such as HTML, CSS, and JavaScript.
- Familiarity with agile methodologies and working in an agile environment.
- Knowledge of accessibility standards and best practices in design.
- Experience in designing for a variety of platforms, including web, mobile, and emerging technologies like AR/VR.
Benefits:
- Competitive salary and performance-based bonuses.
- Comprehensive health, dental, and vision insurance plans.
- Flexible work hours and remote work options.
- Professional development opportunities, including workshops and conferences.
- Generous paid time off and holiday schedule.
- Collaborative and inclusive company culture with regular team-building activities.

In [33]:
#test if the model works
model = 'deepseek-r1:8b_vram'
response = ollama.chat(
    model=model,  
    messages=[
        {'role': 'user', 'content': 'Write in 50 words for the movie Breaking Bad.'},
    ]
)

response['message']['content']

'<think>\nOkay, so I need to write a 50-word summary for the movie "Breaking Bad." Hmm, where do I start? I remember that "Breaking Bad" is about Walter White, right? He\'s a high school chemistry teacher who gets into some serious trouble financially. So he decides to make money by manufacturing and selling drugs. \n\nI think the key elements are his transformation from a普通人到犯罪分子。He starts with making meth in his garage, but it escalates into something bigger. There\'s also his partner Jesse, who is kind of a sidekick but has his own issues. The show talks about morality and the consequences of Walter\'s actions.\n\nWait, should I mention specific characters or just stick to the main plot? Maybe keep it general since it\'s a 50-word summary. Also, I need to make sure it\'s concise. Let me think about the structure: introduce Walter White, his background, his problem, his decision to enter the drug trade, and the consequences or themes involved.\n\nHmm, maybe something like: "Breaking 

### Now we will generate the text files from the CVs and jobs with the help of the model.

In [34]:
def summary_cv_and_write_files(cvs, amount = 500):
    model = 'deepseek-r1:8b_vram'
    prompt = """
You are a CV-to-JSON converter. Transform input CVs into JSON format following these rules:

1. PRESERVE THESE KEYS WITH EXACT TEXT:
   - "Name" (string)
   - "Technical Skills" (array)
   - "Education" (array)
   - "Foreign Languages" (array)
   - "Certifications" (array)
   - "Work Experience" (array, if exists)

2. PROCESS PROJECTS:
   "Project Experience" (object) containing:
   - "Hidden Skills" (array of 3-5 inferred skills)
   - "Technologies Used" (array of explicit tech items)

3. OUTPUT EXAMPLE:
{
  "Name": "Joh Doe",
  "Technical Skills": ["JavaScript", "React", "TypeScript", "Java", "Spring Boot", "AWS", "Docker", "SQL", "PostgreSQL"],
  "Foreign Languages": ["English", "Romanian"],
  "Education": [
    {
      "University Name": "MIT",
      "Program Duration": "2020-2024",
      "Degree Name": "BSc Computer Science"
    }
  ],
  "Certifications": [
    {
      "Title": "AWS Certified Machine Learning – Specialty",
      "Issuing Authority": "Amazon Web Services"
    },
    {
      "Title": "Data Science Professional",
      "Issuing Authority": "Data Science Council"
    }
  ],
  "Project Experience": [
    {
      "Title": "Machine Learning Model Deployment on AWS SageMaker",
      "Hidden Skills": ["Cloud Platforms Expertise", "Containerization Techniques", "CI/CD Pipeline Automation"],
      "Technologies Used": ["Python", "TensorFlow", "AWS SageMaker", "Docker", "Jenkins"]
    },
    {
      "Title": "Interactive Web Application Development",
      "Hidden Skills": ["Responsive Web Development", "UI/UX Design and Prototyping", "Database Integration"],
      "Technologies Used": ["JavaScript", "React.js", "Figma", "PostgreSQL"]
    }
  ]
}

4. SPECIAL RULES:
   - Omit "Project Experience" key entirely if no projects
   - Keep original dates/company names in "Work Experience"
   - Maintain exact certification/language wording
   - Only use double quotes, no trailing commas
   - No additional fields/comments
   
DO NOT ADD ANYTHING ELSE, STRICTLY FOLLOW THE OUTPUT EXAMPLE.
The text:
"""
    if not os.path.exists('text_files'):
        os.makedirs('text_files', exist_ok=True)
    
    if not os.path.exists('text_files/candidates'):
        os.makedirs('text_files/candidates', exist_ok=True)

    for index, (key, value) in enumerate(cvs.items()):
        if index >= amount:
            break

        start_time = time.time()
        print(f'Generating summary for {key}')
        response = ollama.chat(
            model=model,
            #options={'keep_alive': '-1'},
            messages=[
                {'role': 'user', 'content': f"{prompt} {value}"},
            ]
        )
        
        # remove the entire <think>...<./think> section
        summary = (re.sub(r'<think\s*>.*?</think\s*>', '', response['message']['content'], flags=re.DOTALL)
                   .replace('*', '')
                   .replace('```json', '')
                   .replace('```', '')
                   .strip()
                   )
        file_name = key.replace('DataSet/cv\\', '').replace('.docx', '').strip() 
        print(f'Writing summary to file: {file_name}')
        try:
            with open(f'text_files/candidates/{file_name}.json', 'w', encoding='utf-8') as f:
                f.write(summary)
        except Exception as e:
            print(f"Error writing file {file_name}: {e}")

        end_time = time.time()
        overall_time = end_time - start_time
        print(f"Overall time for {file_name}: {overall_time:.2f} seconds")

In [35]:
def summary_jobs_and_write_files(jobs, amount = 500):
    model = 'deepseek-r1:8b_vram'
    prompt = """Act as a job post parser. Analyze the provided job description and return a structured JSON output in the following format:  
{  
  "Type": "[junior/senior/etc] (extracted from job title)",  
  "Role": "[Job Role]",  
  "Company": "[Company Name]",  
  "Responsibilities": [  
    "Summarized responsibility 1",  
    "Summarized responsibility 2",  
    "..."  
  ],  
  "Required": [  
    "Key requirement 1",  
    "Key requirement 2",  
    "... (bullet points)"  
  ],  
  "Skills": {  (for example)
    "Cloud": ["AWS", "Azure", "..."],  
    "Devops": ["Docker", "Jenkins", "..."],  
    "Programming": ["Python", "Java", "..."],
    ...
  },  
  "Benefits": [  
    "Benefit 1 (summarized or verbatim)",  
    "Benefit 2",  
    "... (exact or condensed)"  
  ]  
}  

Follow these rules:  
1. Responsibilities: Summarize key tasks into 4-5 concise bullet points.  
2. Required: List qualifications directly from the "Required Qualifications" section OR SIMILAR.  
3. Skills: Extract and subcategorize skills from the job description. for example: Cloud: ["AWS", "Azure", "..."]
4. Benefits: Retain exact wording or summarize briefly while preserving critical details.  

Return ONLY VALID JSON with no additional text.  

Job Post:  
"""
    if not os.path.exists('text_files'):
        os.makedirs('text_files', exist_ok=True)
    
    if not os.path.exists('text_files/jobs'):
        os.makedirs('text_files/jobs', exist_ok=True)

    
    for index, (key, value) in enumerate(jobs.items()):
        if index >= amount:
            break
            
        print(f'Generating summary for {key}')
        start_time = time.time()
        response = ollama.chat(
            model=model,
            #options={'keep_alive': '-1'},
            messages=[
                {'role': 'user', 'content': f"{prompt} {value}"},
            ]
        )
        
        # remove the entire <think>...<./think> section
        summary = (re.sub(r'<think\s*>.*?</think\s*>', '', response['message']['content'], flags=re.DOTALL)
                   .replace('*', '')
                   .replace('```json', '')
                   .replace('```', '')
                   .strip()
                   )
        file_name = key.replace('DataSet/job\\', '').replace('.docx', '').strip() 
        print(f'Writing summary to file: {file_name}')
        try:
            with open(f'text_files/jobs/{file_name}.json', 'w', encoding='utf-8') as f:
                f.write(summary)
        except Exception as e:
            print(f"Error writing file {file_name}: {e}")

        end_time = time.time()
        overall_time = end_time - start_time
        print(f"Overall time for {file_name}: {overall_time:.2f} seconds")

In [36]:
#summary_cv_and_write_files(cvs, amount=5)
summary_jobs_and_write_files(jobs, amount=10)

Generating summary for  job_description_100_UIUX Designer.docx
Writing summary to file: job_description_100_UIUX Designer
Overall time for job_description_100_UIUX Designer: 26.82 seconds
Generating summary for  job_description_10_Tech Lead.docx
Writing summary to file: job_description_10_Tech Lead
Overall time for job_description_10_Tech Lead: 74.47 seconds
Generating summary for  job_description_11_Product Owner.docx
Writing summary to file: job_description_11_Product Owner
Overall time for job_description_11_Product Owner: 29.27 seconds
Generating summary for  job_description_12_Tech Lead.docx
Writing summary to file: job_description_12_Tech Lead
Overall time for job_description_12_Tech Lead: 77.22 seconds
Generating summary for  job_description_13_Tech Lead.docx
Writing summary to file: job_description_13_Tech Lead
Overall time for job_description_13_Tech Lead: 35.30 seconds
Generating summary for  job_description_14_Backend Developer.docx
Writing summary to file: job_description_1